# Influence Functions: Training Data Attribution

This notebook demonstrates IRTK's `influence_functions` module for tracing model
behaviours back to training examples. We cover:

1. **Influence scores** – which training examples most affect a prediction
2. **Ablation curves** – metric degradation as influential data is removed
3. **Feature-level attribution** – mapping training influence to specific features
4. **Counterfactual training effects** – simulating removal of a training example

## Why This Matters

Influence functions attribute model behavior back to specific training examples — identifying which training data points most influenced a prediction. This connects mechanistic understanding (which components produce the output) to data attribution (which data taught the model this behavior).

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.influence_functions import (
    compute_influence_scores,
    influence_ablation_curve,
    training_example_attribution,
    influence_to_feature,
    counterfactual_training_effect,
)

In [ ]:
# Build a small random model for demonstration
cfg = HookedTransformerConfig(
    n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

# Test and training token sequences
test_tokens = jnp.array([0, 1, 2, 3, 4, 5])
train_seqs = [
    jnp.array([0, 1, 2, 3]),
    jnp.array([4, 5, 6, 7]),
    jnp.array([8, 9, 10, 11]),
    jnp.array([0, 2, 4, 6]),
]
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")
print(f"Test: {test_tokens.tolist()}, Train: {len(train_seqs)} examples")

## 1. Compute Influence Scores

Estimate which training examples most influence the model's prediction of a
particular token. Uses embedding similarity as a proxy for the full Hessian.

In [ ]:
result = compute_influence_scores(model, test_tokens, train_seqs, target_token=0)

print("Influence scores:", result["influence_scores"].round(4))
print("\nTop influential (idx, score):")
for idx, score in result["top_influential"]:
    print(f"  Train example {idx}: influence = {score:.4f}")
print(f"\nTotal positive influence: {result['total_positive_influence']:.4f}")
print(f"Total negative influence: {result['total_negative_influence']:.4f}")

## 2. Influence Ablation Curve

Progressively remove the most influential training examples (via embedding
ablation) and track how the metric degrades.

In [ ]:
def metric_fn(logits):
    return float(logits[-1, 0])

curve = influence_ablation_curve(model, test_tokens, train_seqs, metric_fn, steps=4)

print("Ablation curve:")
for n, m in zip(curve["n_removed"], curve["metrics"]):
    print(f"  Removed {n} examples -> metric = {m:.4f}")
print(f"\nBaseline metric: {curve['baseline_metric']:.4f}")
print(f"Degradation rate: {curve['degradation_rate']:.4f} per example")

## 3. Training Example Attribution

Attribute a specific feature dimension at a hook point to training examples.

In [ ]:
attr = training_example_attribution(
    model, test_tokens, train_seqs, "blocks.0.hook_resid_post", dimension=0
)

print(f"Test activation at dim 0: {attr['test_activation']:.4f}")
print(f"Attributions: {attr['attributions'].round(4)}")
print("\nTop examples:")
for idx, score in attr["top_examples"][:3]:
    print(f"  Train {idx}: attribution = {score:.4f}")

## 4. Influence to Feature

Map training influence to individual feature dimensions at a hook point.

In [ ]:
feat = influence_to_feature(model, test_tokens, train_seqs, "blocks.0.hook_resid_post")

print(f"Feature influences shape: {feat['feature_influences'].shape}")
print(f"Most influenced feature: dim {feat['most_influenced_feature']}")
print(f"Per-feature totals (first 8): {feat['per_feature_total'][:8].round(4)}")

## 5. Counterfactual Training Effect

Simulate removing a single training example's influence by projecting out its
direction from the test representation.

In [ ]:
cf = counterfactual_training_effect(
    model, test_tokens, train_seqs[0], metric_fn
)

print(f"Original metric:       {cf['original_metric']:.4f}")
print(f"Counterfactual metric: {cf['counterfactual_metric']:.4f}")
print(f"Effect (orig - cf):    {cf['effect']:.4f}")
print(f"Alignment (cosine):    {cf['alignment']:.4f}")